In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: O Funcție Qiskit de Qedma
> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă doar utilizatorilor din planurile IBM Quantum&reg; Premium, Flex și On-Prem (prin API-ul IBM Quantum Platform). Acestea se află în stadiu de previzualizare și pot suferi modificări.

## Prezentare generală
Deși unitățile de procesare cuantică s-au îmbunătățit considerabil în ultimii ani, erorile cauzate de zgomot și imperfecțiunile hardware-ului existent rămân o provocare centrală pentru dezvoltatorii de algoritmi cuantici. Pe măsură ce domeniul se apropie de calcule cuantice la scară utilitară care nu pot fi verificate clasic, soluțiile pentru eliminarea zgomotului cu acuratețe garantată devin din ce în ce mai importante. Pentru a depăși această provocare, Qedma a dezvoltat Quantum Error Suppression and Error Mitigation (QESEM), integrat fără întreruperi pe IBM Quantum Platform ca [Funcție Qiskit](/guides/functions).

Cu QESEM, utilizatorii pot rula circuitele lor cuantice pe QPU-uri cu zgomot pentru a obține rezultate precise, fără erori, cu costuri de timp QPU extrem de eficiente, aproape de limitele fundamentale. Pentru a realiza acest lucru, QESEM utilizează o suită de metode proprietare dezvoltate de Qedma pentru caracterizarea și reducerea erorilor. Tehnicile de reducere a erorilor includ optimizarea porților, transpilarea conștientă de zgomot, suprimarea erorilor (ES) și atenuarea nebiasată a erorilor (EM). Prin această combinație de metode bazate pe caracterizare, utilizatorii pot obține rezultate fiabile, fără erori, pentru circuite cuantice generice de volum mare, deblocând aplicații care nu ar putea fi realizate altfel.

Pentru o descriere completă a componentelor de bază, precum și o demonstrație la scară utilitară, consultați articolul [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Descriere
Poți folosi funcția QESEM de Qedma pentru a estima și executa cu ușurință circuitele tale cu suprimare și atenuare a erorilor, obținând volume de circuit mai mari și acuratețe superioară. Pentru a utiliza QESEM, furnizezi un circuit cuantic, un set de observabile de măsurat, o acuratețe statistică țintă pentru fiecare observabilă și un QPU ales. Înainte de a rula circuitul la acuratețea țintă, poți estima timpul QPU necesar pe baza unui calcul analitic care nu necesită execuția circuitului. Odată ce ești mulțumit de estimarea timpului QPU, poți executa circuitul cu QESEM.

Când execuți un circuit, QESEM rulează un protocol de caracterizare a dispozitivului adaptat circuitului tău, producând un model de zgomot fiabil pentru erorile care apar în circuit. Pe baza caracterizării, QESEM implementează mai întâi transpilarea conștientă de zgomot pentru a mapa circuitul de intrare pe un set de qubiți și porți fizice, minimizând zgomotul care afectează observabila țintă. Acestea includ porțile disponibile nativ (CX/CZ pe dispozitivele IBM&reg;), precum și porți suplimentare optimizate de QESEM, formând setul extins de porți al QESEM. QESEM rulează apoi un set de circuite ES și EM bazate pe caracterizare pe QPU și colectează rezultatele măsurătorilor acestora. Acestea sunt apoi post-procesate clasic pentru a furniza o valoare de așteptare nebiasată și o marjă de eroare pentru fiecare observabilă, corespunzătoare acurateței solicitate.

![Prezentare generală Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
S-a demonstrat că QESEM oferă rezultate cu acuratețe ridicată pentru o varietate de aplicații cuantice și pe cele mai mari volume de circuit realizabile astăzi. QESEM oferă următoarele funcționalități orientate către utilizator, demonstrate în secțiunea de benchmark-uri de mai jos:
-	**Acuratețe garantată:** QESEM produce estimări nebiasate pentru valorile de așteptare ale observabilelor. Metoda sa EM este echipată cu garanții teoretice care — împreună cu caracterizarea de ultimă generație a Qedma — asigură că atenuarea converge către ieșirea circuitului fără zgomot, cu o acuratețe specificată de utilizator. Spre deosebire de multe metode EM euristice predispuse la erori sistematice sau biasuri, acuratețea garantată a QESEM este esențială pentru asigurarea unor rezultate fiabile în circuite și observabile cuantice generice.
-	**Scalabilitate pe QPU-uri mari:** Timpul QPU al QESEM depinde de volumele circuitelor, dar este altfel independent de numărul de qubiți. Qedma a demonstrat QESEM pe cele mai mari dispozitive cuantice disponibile astăzi, inclusiv dispozitivele IBM Quantum Eagle de 127 qubiți și Heron de 133 qubiți.
-	**Independent de aplicație:** QESEM a fost demonstrat pe o varietate de aplicații, inclusiv simulare Hamiltoniană, VQE, QAOA și estimare de amplitudine. Utilizatorii pot introduce orice circuit cuantic și observabilă de măsurat și pot obține rezultate precise, fără erori. Singurele limitări sunt dictate de specificațiile hardware-ului și de timpul QPU alocat, care determină volumele de circuit accesibile și acuratețele ieșirii. Spre deosebire de aceasta, multe soluții de reducere a erorilor sunt specifice aplicației sau implică euristici necontrolate, ceea ce le face inaplicabile pentru circuite și aplicații cuantice generice.
-  **Set extins de porți:** QESEM suportă porți cu unghiuri fracționare și furnizează porți $Rzz(\theta)$ cu unghiuri fracționare optimizate de Qedma pe dispozitivele IBM Quantum Eagle. Acest set extins de porți permite o compilare mai eficientă și deblochează volume de circuit mai mari cu un factor de până la 2 față de compilarea implicită CX/CZ.
-	**Observabile multibase:** QESEM suportă observabile de intrare compuse din mai multe șiruri Pauli necomutante, cum ar fi Hamiltonieni generici. Alegerea bazelor de măsurare și optimizarea alocării resurselor QPU (shot-uri și circuite) sunt apoi efectuate automat de QESEM pentru a minimiza timpul QPU necesar la acuratețea solicitată. Această optimizare, care ține cont de fidelitățile hardware-ului și de ratele de execuție, îți permite să rulezi circuite mai profunde și să obții acuratețe mai mare.
## Benchmark-uri
QESEM a fost testat pe o gamă largă de cazuri de utilizare și aplicații. Următoarele exemple te pot ajuta să evaluezi ce tipuri de sarcini de lucru poți rula cu QESEM.

O figură de merit cheie pentru cuantificarea dificultății atât a atenuării erorilor, cât și a simulării clasice pentru un circuit și o observabilă dată este **volumul activ**: numărul de porți CNOT care afectează observabila în circuit. Volumul activ depinde de adâncimea și lățimea circuitului, de greutatea observabilei și de structura circuitului, care determină conul de lumină al observabilei. Pentru detalii suplimentare, consultați prezentarea de la [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM oferă valoare deosebit de mare în regimul cu volum mare, furnizând rezultate fiabile pentru circuite și observabile generice.

![Volum activ](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplicație    | Număr de qubiți | Dispozitiv | Descrierea circuitului | Acuratețe | Timp total | Utilizare runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuit VQE  | 8              | Eagle (r3) | 21 straturi totale, 9 baze de măsurare, lanț 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 straturi unice x 3 pași, topologie 2D heavy-hex                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 straturi unice x 8 pași, topologie 2D heavy-hex                     | 97%      | 116 min      | 23 min          |
| Simulare Hamiltoniană Trotterizată   | 40  | Eagle (r3)            | 2 straturi unice x 10 pași Trotter, lanț 1D                    | 97%      | 3 ore     | 25 min         |
| Simulare Hamiltoniană Trotterizată   | 119  | Eagle (r3)           | 3 straturi unice x 9 pași Trotter, topologie 2D heavy-hex                    | 95%      | 6,5 ore     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 straturi unice x 15 pași, topologie 2D heavy-hex                    | 99%      | 52 min             | 9 min           |

Acuratețea este măsurată aici relativ la valoarea ideală a observabilei: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, unde '$\epsilon$' este precizia absolută a atenuării (stabilită de intrarea utilizatorului), iar $\langle O \rangle_{ideal}$ este observabila la circuitul fără zgomot.
„Utilizarea runtime" măsoară utilizarea benchmark-ului în modul batch (suma utilizărilor job-urilor individuale), în timp ce „timpul total" măsoară utilizarea în modul sesiune (timpul de perete al experimentului), care include timpii clasici și de comunicație suplimentari. QESEM este disponibil pentru execuție în ambele moduri, astfel încât utilizatorii să poată face cel mai bun uz al resurselor disponibile.

Circuitele Kicked Ising de 28 qubiți simulează Cvazicristalul în Timp Discret studiat de Shinjo et al. (vezi [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) și [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) pe trei bucle conectate ale ibm_kawasaki. Parametrii circuitului folosiți aici sunt $(\theta_x, \theta_z) = (0.9 \pi, 0)$, cu o stare inițială feromagnetică $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Observabila măsurată este valoarea absolută a magnetizării $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Experimentul Kicked Ising la scară utilitară a fost rulat pe cei mai buni 136 qubiți ai ibm_fez; acest benchmark particular a fost rulat la unghiul Clifford $(\theta_x, \theta_z) = (\pi, 0)$, la care volumul activ crește lent cu adâncimea circuitului, ceea ce — împreună cu fidelitățile ridicate ale dispozitivului — permite acuratețe ridicată la un timp de execuție scurt.

Circuitele de simulare Hamiltoniană Trotterizată sunt pentru un model Ising cu câmp transversal la unghiuri fracționare: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ și respectiv $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ (vezi [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Circuitul la scară utilitară a fost rulat pe cei mai buni 119 qubiți ai ibm_brisbane, în timp ce experimentul de 40 qubiți a fost rulat pe cel mai bun lanț disponibil. Acuratețea este raportată pentru magnetizare; rezultate cu acuratețe ridicată au fost obținute și pentru observabile de greutate mai mare.

Circuitul VQE a fost dezvoltat împreună cu cercetători de la Centrul pentru Tehnologie și Aplicații Cuantice de la Deutsches Elektronen-Synchrotron (DESY). Observabila țintă a fost un Hamiltonian constând dintr-un număr mare de șiruri Pauli necomutante, subliniind performanța optimizată a QESEM pentru observabile multi-bază. Atenuarea a fost aplicată unui ansatz optimizat clasic; deși aceste rezultate sunt încă nepublicate, rezultate de aceeași calitate vor fi obținute pentru diferite circuite cu proprietăți structurale similare.
## Primii pași
Autentifică-te folosind [cheia ta API IBM Quantum Platform](http://quantum.cloud.ibm.com/) și selectează Funcția Qiskit QESEM după cum urmează. (Acest fragment de cod presupune că ți-ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul local.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Exemplu
Pentru a începe, încearcă acest exemplu de bază pentru estimarea timpului QPU necesar pentru a rula QESEM pentru un `pub` dat:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Următorul exemplu execută un job QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Poți folosi API-urile familiare Qiskit Serverless pentru a verifica starea sarcinii de lucru a Funcției tale Qiskit sau pentru a returna rezultatele:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Parametrii funcției
| Nume |  Tip | Descriere | Obligatoriu | Implicit |  Exemplu |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) | Acesta este intrarea principală. `Pub`-ul conține 2-4 elemente: un Circuit, unul sau mai multe observabile, 0 sau un singur set de valori ale parametrilor și o precizie opțională. Dacă nu s-a specificat o precizie, se va folosi `default_precision` din `options` |  Da|  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string| Numele Backend-ului de utilizat |Nu | QESEM va alege cel mai puțin ocupat dispozitiv raportat de IBM | `"ibm_fez"`|
| `instance` | string|  Numele resursei cloud al instanței de utilizat, în acel format |  Nu |  N/A | `"CRN"`  |
| `options` | dictionary | Opțiuni de intrare. Consultă secțiunea **Options** pentru mai multe detalii. | Nu |  Consultă secțiunea **Options** pentru detalii.    |  `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}`  |

### Options

| Opțiune |  Valori posibile | Descriere | Implicit |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"`  / `"empirical"` / None  | Când este setat, job-ul QESEM va calcula doar estimarea timpului de QPU. Consultă descrierea de mai jos pentru detalii suplimentare. Când este setat la None, circuitul va fi executat cu QESEM | None |
|`default_precision` | 0 < float | Se va aplica `pubs`-urilor care nu au precizie specificată. Precizia indică eroarea acceptabilă asupra valorilor de așteptare ale observabilelor în valoare absolută. Mai precis, durata de rulare QPU pentru mitigare va fi determinată astfel încât să furnizeze valori de ieșire pentru toate observabilele de interes care se încadrează într-un interval de încredere `1σ` față de precizia țintă. Dacă sunt furnizate mai multe observabile, mitigarea va rula până când precizia țintă este atinsă pentru fiecare dintre observabilele de intrare. | 0.02|
|`max_execution_time` | 0 < întreg < 28.800 (8 ore)| Îți permite să limitezi timpul QPU, specificat în secunde, utilizat pentru întregul proces QESEM. Consultă detalii suplimentare mai jos.| 3.600 (o oră)|
| `transpilation_level` | 0 / 1 | Consultă descrierea de mai jos | 1|
| `execution_mode` | `"session"` / `"batch"` | Consultă descrierea de mai jos | "batch"|

> **Caution:** Estimarea timpului QPU variază de la un Backend la altul. Prin urmare, când execuți funcția QESEM, asigură-te că o rulezi pe același Backend care a fost selectat la obținerea estimării timpului QPU.

> **Note:** QESEM își va încheia rularea când atinge precizia țintă sau când atinge `max_execution_time`, oricare survine mai întâi.

- `estimate_time_only` - Acest indicator le permite utilizatorilor să obțină o estimare a timpului QPU necesar pentru a executa circuitul cu QESEM.
    - Dacă este setat la `"analytical"`, se calculează o limită superioară a timpului QPU fără a consuma nicio utilizare QPU. Această estimare are o rezoluție de 30 de minute (de exemplu, 30 de minute, 60 de minute, 90 de minute etc.). Este de obicei pesimistă și poate fi obținută doar pentru observabile Pauli unice sau sume de Pauli fără suporturi care se intersectează (de exemplu, Z0+Z1). Este utilă în principal pentru compararea nivelurilor de complexitate ale diferiților parametri furnizați de utilizator (circuit, acuratețe etc.).
    - Pentru a obține o estimare mai precisă a timpului QPU, setează acest indicator la `"empirical"`. Deși această opțiune necesită rularea unui număr mic de circuite, oferă o estimare a timpului QPU semnificativ mai precisă. Această estimare are o rezoluție de 5 minute (de exemplu, 20 de minute, 25 de minute, 30 de minute etc.). Utilizatorul poate alege să ruleze estimarea empirică a timpului fie în modul batch, fie în modul session. Pentru mai multe detalii, consultă descrierea `execution_mode`. De exemplu, în modul batch, estimarea empirică a timpului va consuma mai puțin de 10 minute de timp QPU.

- `max_execution_time`: Îți permite să limitezi timpul QPU, specificat în secunde, utilizat pentru întregul proces QESEM. Deoarece timpul QPU final necesar pentru a atinge acuratețea țintă este determinat dinamic în timpul job-ului QESEM, acest parametru îți permite să limitezi costul experimentului. Dacă timpul QPU determinat dinamic este mai scurt decât cel alocat de utilizator, acest parametru nu va afecta experimentul. Parametrul `max_execution_time` este deosebit de util în cazurile în care estimarea analitică a timpului furnizată de QESEM înainte de începerea job-ului este prea pesimistă și utilizatorul dorește oricum să inițieze un job de mitigare. După atingerea limitei de timp, QESEM încetează să trimită noi circuite. Circuitele deja trimise continuă să ruleze (astfel că timpul total poate depăși limita cu până la 30 de minute), iar utilizatorul primește rezultatele procesate din circuitele care au rulat până în acel moment. Dacă dorești să aplici o limită de timp QPU mai scurtă decât estimarea analitică a timpului, consultați Qedma pentru a obține o estimare a acurateței realizabile în limita de timp.

- `transpilation_level`: După ce un circuit este trimis la QESEM, acesta pregătește automat mai multe transpilări alternative ale circuitului și o alege pe cea care minimizează timpul QPU. De exemplu, transpilările alternative ar putea utiliza porți RZZ fracționare optimizate de Qedma pentru a reduce adâncimea circuitului. Desigur, toate transpilările sunt echivalente cu circuitul de intrare din punctul de vedere al ieșirii ideale. Pentru un control mai mare asupra transpilării circuitului, setează nivelul de transpilare în `options`. În timp ce `"transpilation_level": 1` corespunde comportamentului implicit descris mai sus, `"transpilation_level": 0` include doar modificările minime necesare circuitului original; de exemplu, „layerification" - organizarea operațiilor circuitului în „straturi" de porți cu doi qubiți simultane. Rețineți că maparea automată pe hardware pe qubiți de înaltă fidelitate se aplică în orice caz.

| transpilation_level | Descriere |
|:-:|:--|
| `1` | Transpilare QESEM implicită. Pregătește mai multe transpilări alternative și o alege pe cea care minimizează timpul QPU. Barierele pot fi modificate în pasul de layerification. |
| `0` | Transpilare minimă: circuitul mitigat va semăna structural îndeaproape cu circuitul de intrare. Circuitele furnizate la nivelul 0 ar trebui să corespundă conectivității dispozitivului și să fie specificate în termenii următoarelor porți: CX, Rzz(α) și porți standard cu un singur qubit (U, x, sx, rz etc.). Barierele vor fi respectate în pasul de layerification. |

- `execution_mode` - Utilizatorul poate alege să ruleze job-ul QESEM fie într-o [sesiune IBM](/guides/execution-modes#session-mode) dedicată, fie în mai multe [loturi IBM](/guides/execution-modes#batch-mode):
    -   **Session Mode**: Sesiunile sunt mai costisitoare, dar duc la un timp mai scurt până la obținerea rezultatelor. Odată ce sesiunea începe, QPU este rezervat exclusiv pentru job-ul QESEM. Calculul utilizării include atât timpul petrecut la execuția QPU, cât și calculele clasice asociate (efectuate de QESEM și IBM). Funcția QESEM Qiskit se ocupă automat de crearea și închiderea sesiunii. Pentru utilizatorii cu acces nelimitat la QPU-uri (de exemplu, configurații on-premises), se recomandă utilizarea modului session pentru o execuție QESEM mai rapidă.
    -   **Batch Mode**: În modul batch, QPU este eliberat în timpul calculelor clasice, ducând la o utilizare mai mică a QPU. Deoarece job-urile batch acoperă de obicei o perioadă mai lungă, există un risc mai mare de derive hardware; QESEM incorporează măsuri pentru a detecta și compensa derivele, menținând fiabilitatea pe parcursul rulărilor extinse.

> **Note:** Operațiunile de barieră sunt utilizate de obicei pentru a specifica straturile de porți cu doi qubiți în circuitele cuantice. La nivelul 0, QESEM păstrează straturile specificate de bariere. La nivelul 1, straturile specificate de bariere sunt considerate ca o alternativă de transpilare la minimizarea timpului QPU.
### Ieșiri
Ieșirea unei funcții Circuit este un [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), care conține două câmpuri:

- Un obiect [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Poate fi indexat direct din `PrimitiveResult`.

- Metadate la nivel de job.

Fiecare `PubResult` conține un câmp `data` și un câmp `metadata`.

- Câmpul `data` conține cel puțin un array de valori de așteptare (`PubResult.data.evs`) și un array de erori standard (`PubResult.data.stds`). Poate conține și mai multe date, în funcție de opțiunile utilizate.

- Câmpul `metadata` conține metadate la nivel PUB (`PubResult.metadata`).

Următorul fragment de cod descrie cum să recuperezi estimarea timpului QPU (când `estimate_time_only` este setat):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

Următorul fragment de cod demonstrează cum să recuperezi rezultatele mitigării (când `estimate_time_only` nu este setat) și metricile de execuție. Acestea conțin date esențiale care permit o înțelegere mai profundă a modului în care diferiți parametri influențează execuția QESEM. Pot fi relevante și la redactarea unui articol bazat pe cercetarea ta.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Obținerea mesajelor de eroare
Dacă starea workload-ului tău este ERROR, folosește job.result() pentru a obține mesajul de eroare, astfel:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>